# Kafka Producer Optimizado con SerpAPI

Este notebook integra la extracción de datos de SerpAPI con un productor de Kafka para generar un stream de datos ricos y útiles para análisis en tiempo real.

## Características:
- Extracción automática de datos de múltiples búsquedas
- Estructuración de datos optimizada para análisis
- Manejo de errores y reintentos
- Metadatos enriquecidos para cada mensaje
- Configuración de paginación para obtener más resultados

In [15]:
import requests
import json
import time
from datetime import datetime
from kafka import KafkaProducer
from kafka.errors import KafkaError
import uuid
from typing import List, Dict, Optional
import logging

# Configurar logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [16]:
class SerpAPIKafkaProducer:
    def __init__(self, api_key: str, kafka_servers: List[str] = ['localhost:9092']):
        self.api_key = api_key
        self.endpoint = "https://serpapi.com/search"
        
        # Configuración del productor Kafka
        self.producer = KafkaProducer(
            bootstrap_servers=kafka_servers,
            value_serializer=lambda v: json.dumps(v, ensure_ascii=False, default=str).encode('utf-8'),
            key_serializer=lambda k: k.encode('utf-8') if k else None,
            acks='all',  # Esperar confirmación de todos los brokers
            retries=3,   # Reintentos automáticos
            max_in_flight_requests_per_connection=1  # Garantizar orden
        )
        
        # Configuraciones por país disponibles
        self.country_configs = {
            'germany': {
                "hl": "es",  # idioma español
                "gl": "de",  # Alemania
                "location": "Germany",
                "country_name": "Germany",
                "country_code": "DE",
                "language": "english",
                "currency": "EUR"
            },
            'usa': {
                "hl": "en",  # idioma inglés
                "gl": "us",  # Estados Unidos
                "location": "United States",
                "country_name": "United States",
                "country_code": "US", 
                "language": "english",
                "currency": "USD"
            },
            'spain': {
                "hl": "es",  # idioma español
                "gl": "es",  # España
                "location": "Spain",
                "country_name": "Spain",
                "country_code": "ES",
                "language": "español",
                "currency": "EUR"
            },
            'mexico': {
                "hl": "es",  # idioma español
                "gl": "mx",  # México
                "location": "Mexico",
                "country_name": "Mexico",
                "country_code": "MX",
                "language": "español",
                "currency": "MXN"
            },
            'ecuador': {
                "hl": "es",  # idioma español
                "gl": "ec",  # Ecuador
                "location": "Ecuador",
                "country_name": "Ecuador",
                "country_code": "EC",
                "language": "español",
                "currency": "USD"
            }
        }
        
        # Configuración inicial (por defecto Alemania)
        self.current_country = 'germany'
        self.base_params = {
            "engine": "google",
            "format": "json",
            "api_key": self.api_key,
            "num": 10,  # Más resultados por búsqueda
        }
        self._update_country_params()
    
    def set_country(self, country_key: str):
        """Cambiar la configuración del país para las búsquedas"""
        if country_key in self.country_configs:
            self.current_country = country_key
            self._update_country_params()
            logging.info(f"País configurado a: {self.country_configs[country_key]['country_name']}")
        else:
            available_countries = list(self.country_configs.keys())
            logging.error(f"País '{country_key}' no disponible. Países disponibles: {available_countries}")
    
    def _update_country_params(self):
        """Actualizar parámetros base con la configuración del país actual"""
        config = self.country_configs[self.current_country]
        self.base_params.update({
            "hl": config["hl"],
            "gl": config["gl"], 
            "location": config["location"]
        })
        
    def get_available_countries(self) -> Dict[str, str]:
        """Obtener lista de países disponibles"""
        return {key: config['country_name'] for key, config in self.country_configs.items()}
    
    def search_serp_api(self, keyword: str, start: int = 0) -> Optional[Dict]:
        """Realizar búsqueda en SerpAPI con manejo de errores"""
        params = self.base_params.copy()
        params["q"] = keyword
        params["start"] = start
        
        try:
            response = requests.get(self.endpoint, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            logging.error(f"Error en la solicitud a SerpAPI: {e}")
            return None
        except json.JSONDecodeError as e:
            logging.error(f"Error al decodificar JSON: {e}")
            return None
    
    def extract_structured_data(self, search_result: Dict, keyword: str) -> List[Dict]:
        """Extraer y estructurar datos de los resultados de búsqueda"""
        structured_data = []
        
        if not search_result:
            return structured_data
        
        # Obtener configuración del país actual
        country_config = self.country_configs[self.current_country]
        
        # Metadatos de la búsqueda con información geográfica
        search_metadata = {
            'search_id': search_result.get('search_metadata', {}).get('id'),
            'search_timestamp': datetime.now().isoformat(),
            'search_keyword': keyword,
            'total_results': search_result.get('search_information', {}).get('total_results', 0),
            'time_taken': search_result.get('search_information', {}).get('time_taken_displayed', 0),
            # 🌍 Información geográfica del país
            'country_info': {
                'country_key': self.current_country,
                'country_name': country_config['country_name'],
                'country_code': country_config['country_code'],
                'language': country_config['language'],
                'currency': country_config['currency'],
                'location_param': country_config['location'],
                'gl_param': country_config['gl'],
                'hl_param': country_config['hl']
            }
        }
        
        # Procesar resultados orgánicos
        organic_results = search_result.get('organic_results', [])
        for result in organic_results:
            structured_item = {
                'id': str(uuid.uuid4()),
                'type': 'organic_result',
                'search_metadata': search_metadata,
                'position': result.get('position'),
                'title': result.get('title'),
                'link': result.get('link'),
                'snippet': result.get('snippet'),
                'source': result.get('source'),
                'date': result.get('date'),
                'highlighted_words': result.get('snippet_highlighted_words', []),
                'missing_keywords': result.get('missing', []),
                'created_at': datetime.now().isoformat()
            }
            structured_data.append(structured_item)
        
        # Procesar búsquedas relacionadas
        related_searches = search_result.get('related_searches', [])
        for related in related_searches:
            structured_item = {
                'id': str(uuid.uuid4()),
                'type': 'related_search',
                'search_metadata': search_metadata,
                'query': related.get('query'),
                'link': related.get('link'),
                'serpapi_link': related.get('serpapi_link'),
                'created_at': datetime.now().isoformat()
            }
            structured_data.append(structured_item)
        
        return structured_data
    
    def send_to_kafka(self, data: List[Dict], topic: str = 'search_analytics'):
        """Enviar datos estructurados a Kafka"""
        for item in data:
            try:
                future = self.producer.send(
                    topic=topic,
                    key=item['id'],
                    value=item
                )
                logging.info(f"Enviado: {item['type']} - {item.get('title', item.get('query', 'N/A'))}")
            except Exception as e:
                logging.error(f"Error enviando a Kafka: {e}")
        
        # Asegurar que todos los mensajes se envíen
        self.producer.flush()
    
    def stream_search_data(self, keywords: List[str], topic: str = 'search_analytics', 
                          pages_per_keyword: int = 2, delay_between_searches: int = 3):
        """Stream continuo de datos de búsqueda a Kafka"""
        
        logging.info(f"Iniciando stream de datos para {len(keywords)} keywords")
        
        for keyword in keywords:
            logging.info(f"Procesando keyword: {keyword}")
            
            # Obtener múltiples páginas para cada keyword
            for page in range(pages_per_keyword):
                start_index = page * 10
                
                logging.info(f"Obteniendo página {page + 1} para '{keyword}'")
                
                search_result = self.search_serp_api(keyword, start_index)
                
                if search_result:
                    structured_data = self.extract_structured_data(search_result, keyword)
                    
                    if structured_data:
                        self.send_to_kafka(structured_data, topic)
                        logging.info(f"Enviados {len(structured_data)} elementos a Kafka")
                    else:
                        logging.warning(f"No se obtuvieron datos para '{keyword}' página {page + 1}")
                else:
                    logging.error(f"Error obteniendo datos para '{keyword}' página {page + 1}")
                
                time.sleep(2)
            
            if keyword != keywords[-1]:
                logging.info(f"Esperando {delay_between_searches} segundos...")
                time.sleep(delay_between_searches)
        
        logging.info("Stream de datos completado")
    
    def close(self):
        """Cerrar el productor Kafka"""
        if self.producer:
            self.producer.close()
            logging.info("Productor Kafka cerrado")


In [17]:
# 🌍 CONFIGURACIÓN MULTI-PAÍS
# Ejemplo de uso del Producer con diferentes países

# Configurar API key
API_KEY = "70754b855634ed741481ef99277328d8228191bb972a8741acd8826d7505d884"

# Crear el producer
serp_kafka = SerpAPIKafkaProducer(api_key=API_KEY)

# Ver países disponibles
print("🌍 Países disponibles:")
for key, name in serp_kafka.get_available_countries().items():
    print(f"   {key}: {name}")

print("\n" + "="*50)

# Ejemplo: Análisis de búsquedas turísticas en diferentes países
keywords_tourism = [
    "hoteles baratos",
    "vuelos económicos", 
    "destinos turísticos",
    "tours guiados"
]

# 🇩🇪 Datos de Alemania (idioma español)
print("🇩🇪 Configurando para Alemania...")
serp_kafka.set_country('germany')
print("✅ Listo para obtener datos de búsqueda en inglés con resultados de Alemania")

# 🇺🇸 Datos de Estados Unidos (idioma inglés)
print("\n🇺🇸 Cambiando a Estados Unidos...")
serp_kafka.set_country('usa')
print("✅ Listo para obtener datos de búsqueda en inglés con resultados de EE.UU.")

# 🇪🇸 Datos de España (idioma español)
print("\n🇪🇸 Cambiando a España...")
serp_kafka.set_country('spain')
print("✅ Listo para obtener datos de búsqueda en español con resultados de España")

# 🇪🇨 Datos de Ecuador (idioma español)
print("\n🇪🇨 Cambiando a Ecuador...")
serp_kafka.set_country('ecuador')
print("✅ Listo para obtener datos de búsqueda en español con resultados de Ecuador")

# 🇲🇽 Datos de México (idioma español)
print("\n🇲🇽 Cambiando a México...")
serp_kafka.set_country('mexico')
print("✅ Listo para obtener datos de búsqueda en español con resultados de México")

print("\n💡 Ahora los datos enviados a Kafka incluirán información del país configurado")


2025-07-24 16:56:18,951 - INFO - <BrokerConnection client_id=kafka-python-producer-7, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: connecting to localhost:9092 [('::1', 9092, 0, 0) IPv6]
2025-07-24 16:56:18,954 - INFO - <BrokerConnection client_id=kafka-python-producer-7, node_id=bootstrap-0 host=localhost:9092 <checking_api_versions_recv> [IPv6 ('::1', 9092, 0, 0)]>: Broker version identified as 2.6
2025-07-24 16:56:18,955 - INFO - <BrokerConnection client_id=kafka-python-producer-7, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Connection complete.
2025-07-24 16:56:18,957 - INFO - <BrokerConnection client_id=kafka-python-producer-5, node_id=1 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Closing connection. 
2025-07-24 16:56:18,958 - INFO - País configurado a: Germany
2025-07-24 16:56:18,959 - INFO - País configurado a: United States
2025-07-24 16:56:18,960 - INFO - País configurado a: Spain
2025-0

🌍 Países disponibles:
   germany: Germany
   usa: United States
   spain: Spain
   mexico: Mexico
   ecuador: Ecuador

🇩🇪 Configurando para Alemania...
✅ Listo para obtener datos de búsqueda en inglés con resultados de Alemania

🇺🇸 Cambiando a Estados Unidos...
✅ Listo para obtener datos de búsqueda en inglés con resultados de EE.UU.

🇪🇸 Cambiando a España...
✅ Listo para obtener datos de búsqueda en español con resultados de España

🇪🇨 Cambiando a Ecuador...
✅ Listo para obtener datos de búsqueda en español con resultados de Ecuador

🇲🇽 Cambiando a México...
✅ Listo para obtener datos de búsqueda en español con resultados de México

💡 Ahora los datos enviados a Kafka incluirán información del país configurado


In [19]:
# 🌍 EJEMPLO PRÁCTICO: ANÁLISIS MULTI-PAÍS DE TURISMO
# Este ejemplo muestra cómo generar datos de diferentes países para análisis comparativo

# CONFIGURAR TU API KEY AQUÍ
API_KEY = "70754b855634ed741481ef99277328d8228191bb972a8741acd8826d7505d884"  # ⚠️ Reemplazar con tu clave real

# Palabras clave de turismo comunes (que funcionan en múltiples idiomas)
keywords_tourism = [
    "hoteles baratos",      # ES: hoteles económicos  
    "vuelos economicos",    # ES: vuelos baratos
    "que visitar",          # ES: lugares turísticos
    "tours guiados",        # ES: tours con guía
    "lugares turisticos",   # ES: lugares turísticos
]

keywords_tourism_en = [
    "cheap hotels",         # EN: hoteles baratos
    "budget flights",       # EN: vuelos económicos  
    "tourist attractions",  # EN: atracciones turísticas
    "guided tours",         # EN: tours guiados
    "tourist places",       # EN: lugares turísticos
]

def run_multi_country_analysis():
    """Ejecutar análisis de turismo en múltiples países"""
    
    # Crear producer
    serp_producer = SerpAPIKafkaProducer(api_key=API_KEY)
    
    print("🌍" + "="*60)
    print("     ANÁLISIS MULTI-PAÍS DE DATOS TURÍSTICOS")
    print("="*60)
    
    # 🇩🇪 ALEMANIA - Mercado europeo (alemán)
    print("\n🇩🇪 GENERANDO DATOS DE ALEMANIA...")
    serp_producer.set_country('germany')
    print(f"✅ Configurado para: {serp_producer.country_configs['germany']['country_name']}")
    print(f"   💰 Moneda: {serp_producer.country_configs['germany']['currency']}")
    print(f"   🗣️ Idioma: {serp_producer.country_configs['germany']['language']}")
    
    # Ejecutar búsquedas para Alemania
    serp_producer.stream_search_data(
        keywords=keywords_tourism[:2],  # Solo 2 keywords para demo
        topic='tourism_analytics',
        pages_per_keyword=1,
        delay_between_searches=2
    )
    
    # 🇲🇽 Datos de México (idioma español)
    print("\n🇲🇽 GENERANDO DATOS DE MÉXICO...")
    serp_producer.set_country('mexico')
    print(f"✅ Configurado para: {serp_producer.country_configs['mexico']['country_name']}")
    print(f"   💰 Moneda: {serp_producer.country_configs['mexico']['currency']}")
    print(f"   🗣️ Idioma: {serp_producer.country_configs['mexico']['language']}")
    
    # Ejecutar búsquedas para México
    serp_producer.stream_search_data(
        keywords=keywords_tourism[:2],  # Solo 2 keywords para demo
        topic='tourism_analytics',
        pages_per_keyword=1,
        delay_between_searches=2
    )
    
    
    # 🇪🇨 ECUADOR - Mercado turístico local (español)
    print("\n🇪🇨 GENERANDO DATOS DE ECUADOR...")
    serp_producer.set_country('ecuador')
    print(f"✅ Configurado para: {serp_producer.country_configs['ecuador']['country_name']}")
    print(f"   💰 Moneda: {serp_producer.country_configs['ecuador']['currency']}")
    print(f"   🗣️ Idioma: {serp_producer.country_configs['ecuador']['language']}")
    
    # Ejecutar algunas búsquedas para Ecuador
    serp_producer.stream_search_data(
        keywords=keywords_tourism[:2],  # Solo 2 keywords para demo
        topic='tourism_analytics',
        pages_per_keyword=1,
        delay_between_searches=2
    )
    
    # 🇺🇸 ESTADOS UNIDOS - Mercado turístico internacional (inglés)  
    print("\n🇺🇸 GENERANDO DATOS DE ESTADOS UNIDOS...")
    serp_producer.set_country('usa')
    print(f"✅ Configurado para: {serp_producer.country_configs['usa']['country_name']}")
    print(f"   💰 Moneda: {serp_producer.country_configs['usa']['currency']}")
    print(f"   🗣️ Idioma: {serp_producer.country_configs['usa']['language']}")
    
    # Ejecutar búsquedas para USA en inglés
    serp_producer.stream_search_data(
        keywords=keywords_tourism_en[:2],  # Términos en inglés
        topic='tourism_analytics',
        pages_per_keyword=1,
        delay_between_searches=2
    )
    
    # 🇪🇸 ESPAÑA - Mercado europeo (español)
    print("\n🇪🇸 GENERANDO DATOS DE ESPAÑA...")
    serp_producer.set_country('spain')
    print(f"✅ Configurado para: {serp_producer.country_configs['spain']['country_name']}")
    print(f"   💰 Moneda: {serp_producer.country_configs['spain']['currency']}")
    print(f"   🗣️ Idioma: {serp_producer.country_configs['spain']['language']}")
    
    # Ejecutar búsquedas para España
    serp_producer.stream_search_data(
        keywords=keywords_tourism[:2],  # Términos en español  
        topic='tourism_analytics',
        pages_per_keyword=1,
        delay_between_searches=2
    )
    
    print("\n🎉 ANÁLISIS COMPLETADO!")
    print("📊 Los datos ahora están en Kafka con información del país")
    print("🌍 Puedes filtrar por país en Grafana: Ecuador, United States, Spain")
    print("💡 Consumer procesará automáticamente la información geográfica")

# Para ejecutar el análisis, descomenta la siguiente línea:
run_multi_country_analysis()


2025-07-24 17:02:01,987 - INFO - <BrokerConnection client_id=kafka-python-producer-9, node_id=bootstrap-0 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: connecting to localhost:9092 [('::1', 9092, 0, 0) IPv6]
2025-07-24 17:02:01,992 - INFO - <BrokerConnection client_id=kafka-python-producer-9, node_id=bootstrap-0 host=localhost:9092 <checking_api_versions_recv> [IPv6 ('::1', 9092, 0, 0)]>: Broker version identified as 2.6
2025-07-24 17:02:01,993 - INFO - <BrokerConnection client_id=kafka-python-producer-9, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Connection complete.
2025-07-24 17:02:01,995 - INFO - País configurado a: Germany
2025-07-24 17:02:01,996 - INFO - Iniciando stream de datos para 2 keywords
2025-07-24 17:02:01,997 - INFO - Procesando keyword: hoteles baratos
2025-07-24 17:02:01,997 - INFO - Obteniendo página 1 para 'hoteles baratos'


🌍============================================================
     ANÁLISIS MULTI-PAÍS DE DATOS TURÍSTICOS

🇩🇪 GENERANDO DATOS DE ALEMANIA...
✅ Configurado para: Germany
   💰 Moneda: EUR
   🗣️ Idioma: english


2025-07-24 17:02:02,597 - INFO - <BrokerConnection client_id=kafka-python-producer-9, node_id=1 host=localhost:9092 <connecting> [IPv6 ('::1', 9092, 0, 0)]>: connecting to localhost:9092 [('::1', 9092, 0, 0) IPv6]
2025-07-24 17:02:02,601 - INFO - <BrokerConnection client_id=kafka-python-producer-9, node_id=1 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Connection complete.
2025-07-24 17:02:02,604 - INFO - <BrokerConnection client_id=kafka-python-producer-9, node_id=bootstrap-0 host=localhost:9092 <connected> [IPv6 ('::1', 9092, 0, 0)]>: Closing connection. 
2025-07-24 17:02:02,616 - INFO - Enviado: organic_result - Hoteles. Reserva de Hoteles Baratos Online
2025-07-24 17:02:02,626 - INFO - Enviado: organic_result - Hoteles | Comparador de hoteles baratos
2025-07-24 17:02:02,637 - INFO - Enviado: organic_result - Hoteles Baratos | Reserva Hoteles cerca de ti - Lastminute
2025-07-24 17:02:02,643 - INFO - Enviado: organic_result - Ofertas de hoteles baratos | Compara y res


🇲🇽 GENERANDO DATOS DE MÉXICO...
✅ Configurado para: Mexico
   💰 Moneda: MXN
   🗣️ Idioma: español


2025-07-24 17:02:09,616 - INFO - Enviado: organic_result - Ofertas de hoteles baratos | Compara y reserva con ...
2025-07-24 17:02:09,626 - INFO - Enviado: organic_result - Hoteles baratos, alojamiento y hospedaje
2025-07-24 17:02:09,641 - INFO - Enviado: organic_result - ¡Hoteles Baratos en México y el Mundo!
2025-07-24 17:02:09,650 - INFO - Enviado: organic_result - trivago.com.mx - Comparar precios de hoteles en todo el mundo
2025-07-24 17:02:09,664 - INFO - Enviado: organic_result - Hoteles.com - Ofertas y descuentos para reservaciones en ...
2025-07-24 17:02:09,668 - INFO - Enviado: organic_result - Los mejores hoteles económicos en Ciudad de México
2025-07-24 17:02:09,680 - INFO - Enviado: organic_result - Hoteles y Alojamientos | Reserva hoteles baratos en ...
2025-07-24 17:02:09,688 - INFO - Enviado: organic_result - Hoteles en Ciudad de México desde $149/noche
2025-07-24 17:02:09,692 - INFO - Enviado: organic_result - Mejores Hoteles baratos en Ciudad de México para 2025
2025-

...


🇺🇸 GENERANDO DATOS DE ESTADOS UNIDOS...
✅ Configurado para: United States
   💰 Moneda: USD
   🗣️ Idioma: english


2025-07-24 17:02:23,545 - INFO - Enviado: organic_result - CheapHotels.com: Official Site
2025-07-24 17:02:23,557 - INFO - Enviado: organic_result - Hotwire: Cheap Hotels, Cars, & Flights | Last Minute Travel ...
2025-07-24 17:02:23,564 - INFO - Enviado: organic_result - 10 Best Cheap Hotels in Houston
2025-07-24 17:02:23,572 - INFO - Enviado: organic_result - 10 Best Cheap Hotels in Detroit
2025-07-24 17:02:23,582 - INFO - Enviado: organic_result - Last Minute Hotel Deals at Great Hotels - HotelTonight
2025-07-24 17:02:23,591 - INFO - Enviado: organic_result - Cheap Hotels - Book cheap hotel rooms worldwide
2025-07-24 17:02:23,602 - INFO - Enviado: organic_result - Hotels: Find Cheap Hotel Deals & Discounts
2025-07-24 17:02:23,615 - INFO - Enviado: organic_result - 10 Best Cheap Hotels in Los Angeles
2025-07-24 17:02:23,624 - INFO - Enviado: organic_result - 10 Best Cheap Hotels in Fort Worth
2025-07-24 17:02:23,641 - INFO - Enviado: related_search - None
2025-07-24 17:02:23,646 - INF


🇪🇸 GENERANDO DATOS DE ESPAÑA...
✅ Configurado para: Spain
   💰 Moneda: EUR
   🗣️ Idioma: español


2025-07-24 17:02:30,509 - INFO - Enviado: organic_result - Hoteles. Reserva de Hoteles Baratos Online
2025-07-24 17:02:30,512 - INFO - Enviado: organic_result - Ofertas de hoteles baratos - Skyscanner
2025-07-24 17:02:30,516 - INFO - Enviado: organic_result - Hoteles. Ofertas de hoteles | Tu hotel en Viajes Carrefour
2025-07-24 17:02:30,523 - INFO - Enviado: organic_result - Hoteles Baratos | Reserva Hoteles cerca de ti - Lastminute
2025-07-24 17:02:30,531 - INFO - Enviado: organic_result - ¡Que Hoteles! HOTELES y más al MEJOR PRECIO.
2025-07-24 17:02:30,537 - INFO - Enviado: organic_result - Hoteles baratos en Madrid desde 52 €. ¡Reserva ya en ...
2025-07-24 17:02:30,538 - INFO - Enviado: organic_result - Hoteles | Comparador de hoteles baratos
2025-07-24 17:02:30,547 - INFO - Enviado: organic_result - Hoteles baratos en Barcelona desde 61 €. ¡Reserva ya en ...
2025-07-24 17:02:30,558 - INFO - Enviado: organic_result - Hoteles baratos y Alquileres vacacionales | cozycozy
2025-07-24 17


🎉 ANÁLISIS COMPLETADO!
📊 Los datos ahora están en Kafka con información del país
🌍 Puedes filtrar por país en Grafana: Ecuador, United States, Spain
💡 Consumer procesará automáticamente la información geográfica


In [ ]:
# Configuración principal
API_KEY = "70754b855634ed741481ef99277328d8228191bb972a8741acd8826d7505d884"
KAFKA_SERVERS = ['localhost:9092']
TOPIC = 'tourism_analytics'

# Keywords relacionadas con turismo y análisis de mercado para Ecuador
TOURISM_KEYWORDS = [
    "destinos emergentes Ecuador 2025",
    "turismo sostenible Ecuador",
    "ecoturismo Galápagos",
    "turismo aventura Ecuador",
    "hoteles boutique Quito",
    "gastronomía ecuatoriana",
    "festivales culturales Ecuador",
    "turismo comunitario Ecuador"
]

# Inicializar el productor optimizado
serp_kafka = SerpAPIKafkaProducer(API_KEY, KAFKA_SERVERS)
print("✅ Productor SerpAPI-Kafka inicializado correctamente")


In [ ]:
# Ejecutar el streaming de datos optimizado
try:
    print("Iniciando streaming de datos de SerpAPI a Kafka...")
    
    serp_kafka.stream_search_data(
        keywords=TOURISM_KEYWORDS,
        topic=TOPIC,
        pages_per_keyword=1,  # 2 páginas por keyword para obtener más datos
        delay_between_searches=3  # 3 segundos entre keywords para evitar rate limiting
    )
    
    print("Streaming completado exitosamente")
    
except KeyboardInterrupt:
    print("Streaming interrumpido por el usuario")
except Exception as e:
    print(f"Error durante el streaming: {e}")
    logging.error(f"Error durante el streaming: {e}")
finally:
    serp_kafka.close()
    print("Productor Kafka cerrado")


## Funciones Avanzadas para Análisis Específicos

Las siguientes funciones te permiten realizar análisis más específicos y personalizados:

### 🔍 Características del Sistema Optimizado:

1. **Datos Estructurados**: Cada mensaje incluye metadatos ricos para análisis
2. **Manejo de Errores**: Reintentos automáticos y logging detallado
3. **Rate Limiting**: Delays configurables para evitar límites de API
4. **Escalabilidad**: Distribución por particiones usando IDs únicos
5. **Flexibilidad**: Búsquedas personalizadas y análisis de tendencias

### 📊 Tipos de Datos Generados:

- **organic_result**: Resultados principales de búsqueda
- **related_search**: Búsquedas relacionadas sugeridas
- **search_metadata**: Metadatos de cada búsqueda realizada

In [ ]:
def analyze_trending_topics(serp_kafka: SerpAPIKafkaProducer, base_topic: str, topic: str = 'trending_analysis'):
    """Analizar temas trending basados en un tópico base"""
    
    print(f"🔍 Analizando trending topics para: {base_topic}")
    
    # Buscar el tópico base
    result = serp_kafka.search_serp_api(base_topic)
    
    if result and 'related_searches' in result:
        related_queries = [r['query'] for r in result['related_searches']]
        
        print(f"📈 Encontrados {len(related_queries)} temas relacionados")
        
        # Usar las búsquedas relacionadas como nuevos keywords
        serp_kafka.stream_search_data(
            keywords=related_queries[:5],  # Top 5 relacionados
            topic=topic,
            pages_per_keyword=1
        )
        
        return related_queries
    return []

def monitor_competition(serp_kafka: SerpAPIKafkaProducer, business_keywords: List[str], topic: str = 'competition_analysis'):
    """Monitorear competencia para keywords de negocio específicos"""
    
    print(f"🏢 Iniciando análisis de competencia para {len(business_keywords)} keywords")
    
    competitive_keywords = []
    for keyword in business_keywords:
        competitive_keywords.extend([
            f"{keyword} precios",
            f"{keyword} ofertas",
            f"{keyword} reviews",
            f"mejor {keyword}"
        ])
    
    print(f"📊 Analizando {len(competitive_keywords)} variaciones competitivas")
    
    serp_kafka.stream_search_data(
        keywords=competitive_keywords,
        topic=topic,
        pages_per_keyword=1
    )

def real_time_search(serp_kafka: SerpAPIKafkaProducer, keyword: str, topic: str = 'real_time_search'):
    """Realizar una búsqueda en tiempo real y enviar a Kafka"""
    
    print(f"⚡ Búsqueda en tiempo real para: {keyword}")
    
    # Realizar búsqueda
    result = serp_kafka.search_serp_api(keyword)
    
    if result:
        # Estructurar datos
        structured_data = serp_kafka.extract_structured_data(result, keyword)
        
        # Enviar a Kafka
        serp_kafka.send_to_kafka(structured_data, topic)
        
        print(f"✅ Enviados {len(structured_data)} elementos para '{keyword}'")
        return structured_data
    else:
        print(f"❌ No se pudieron obtener datos para '{keyword}'")
        return []

# Ejemplo de uso para análisis de tendencias
# trending_topics = analyze_trending_topics(serp_kafka, "turismo Ecuador 2025")
# print(f"Temas trending encontrados: {trending_topics}")


## 📋 Estructura de Datos Generados

Los datos enviados a Kafka tienen la siguiente estructura optimizada para análisis:

```json
{
  "id": "uuid único",
  "type": "organic_result | related_search",
  "search_metadata": {
    "search_id": "id de búsqueda de SerpAPI",
    "search_timestamp": "timestamp ISO 8601",
    "search_keyword": "keyword buscado",
    "total_results": "número total de resultados",
    "time_taken": "tiempo de búsqueda en segundos"
  },
  "title": "título del resultado",
  "link": "URL del resultado",
  "snippet": "descripción del resultado",
  "source": "fuente del contenido",
  "date": "fecha de publicación",
  "highlighted_words": ["palabras destacadas"],
  "missing_keywords": ["keywords ausentes"],
  "created_at": "timestamp de creación del registro"
}
```

## 📈 Casos de Uso para Análisis de Datos

Esta estructura permite realizar diversos análisis:

### 🎯 Análisis de Mercado
- **Trending topics** por fecha y región
- **Análisis de sentimiento** en snippets
- **Monitoreo de competencia** en tiempo real
- **Identificación de fuentes** más relevantes

### 🔍 Análisis de Búsquedas
- **Tracking de palabras clave** emergentes
- **Patrones de búsqueda** estacionales
- **Análisis de relacionalidad** entre temas
- **Detección de oportunidades** de contenido

### 📊 Métricas de Rendimiento
- **Volumen de resultados** por keyword
- **Tiempo de respuesta** de búsquedas
- **Efectividad de keywords** específicos
- **Análisis de posicionamiento** de contenido
